# Local Import Requirements

In [1]:
import os
os.environ['DRJIT_LIBLLVM_PATH'] = '/usr/lib/x86_64-linux-gnu/libLLVM-20.so'
import drjit as drjit

%env DRJIT_LIBLLVM_PATH="/usr/lib/x86_64-linux-gnu/libLLVM-20.so"

env: DRJIT_LIBLLVM_PATH="/usr/lib/x86_64-linux-gnu/libLLVM-20.so"


In [2]:
import numpy as np
import tensorflow as tf

import sionna
# Import or install Sionna
try:
    import sionna.rt
except ImportError as e:
    import os
    os.system("pip install sionna-rt")
    import sionna.rt

# Other imports
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
from scipy import constants

no_preview = False # Toggle to False to use the preview widget
import mitsuba as mi
# mi.set_variant("cuda_rgb")
# Import relevant components from Sionna RT
from sionna.rt import load_scene, PlanarArray, Transmitter, Receiver, Camera,\
                      PathSolver, RadioMapSolver, subcarrier_frequencies, LambertianPattern

import gc

2026-04-02 17:14:46.912999: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [5]:
# import inspect
# import sionna.rt.path_solvers.sb_candidate_generator as sb
# print(inspect.getfile(sb))

In [3]:
print(tf.config.list_physical_devices())
print(tf.config.list_physical_devices("GPU"))

[PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU'), PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


# Calculate Transmitter Positions, Beam Angles

In [7]:
# 250 m 
d = 250
rx_pos_sionna = np.array([300, 100, 10])

SIONNA_20_DEG = rx_pos_sionna + [d*np.cos(20*np.pi/180), 0, d*np.sin(20*np.pi/180)]
SIONNA_50_DEG = rx_pos_sionna + [d*np.cos(50*np.pi/180), 0, d*np.sin(50*np.pi/180)]
SIONNA_90_DEG = rx_pos_sionna + [0, 0, d]

sionna_250m = [SIONNA_20_DEG, SIONNA_50_DEG, SIONNA_90_DEG]


# 1km
d = 1000

SIONNA_20_DEG = rx_pos_sionna + [d*np.cos(20*np.pi/180), 0, d*np.sin(20*np.pi/180)]
SIONNA_50_DEG = rx_pos_sionna + [d*np.cos(50*np.pi/180), 0, d*np.sin(50*np.pi/180)]
SIONNA_90_DEG = rx_pos_sionna + [0, 0, d]

sionna_1km = [SIONNA_20_DEG, SIONNA_50_DEG, SIONNA_90_DEG]

# LEO km 
slants = [2_703_801, 698_920, 550_000]

SIONNA_20_DEG = rx_pos_sionna + [slants[0]*np.cos(20*np.pi/180), 0, slants[0]*np.sin(20*np.pi/180)]
SIONNA_50_DEG = rx_pos_sionna + [slants[1]*np.cos(50*np.pi/180), 0, slants[1]*np.sin(50*np.pi/180)]
SIONNA_90_DEG = rx_pos_sionna + [0, 0, slants[2]]

sionna_LEO = [SIONNA_20_DEG, SIONNA_50_DEG, SIONNA_90_DEG]

In [8]:
# Calculate beam angle to illuminate scene as function of elevation angle(radians)
angle_250 = np.arctan(50/250)

angle_1k = np.arctan(50/1000)

angle_leo = np.arctan(50/550_000)


In [9]:
print(angle_250)
print(angle_1k)
print(angle_leo)

0.19739555984988078
0.049958395721942765
9.090909065865264e-05


# Helper Functions

In [4]:
def mean_and_variance(taus, amps):
    tau_m = []
    tau_rms = []

    for i, t in enumerate(taus):
        t = np.asarray(t)
        a = np.asarray(amps[i])

        idx = np.argsort(t)
        t = t[idx]
        a = a[idx]

        tau0 = t[0]

        P = np.abs(a)**2          # power weights
        den = np.sum(P)

        dt = t - tau0
        
        m = np.dot(dt, P) / den
        v = np.dot((dt - m)**2, P) / den

        tau_m.append(m)
        tau_rms.append(np.sqrt(v))

    return tau_m, tau_rms

In [5]:
def calculate_path_distance(tx_pos, rx_pos, points, objects):
    """
    returns distance for path, number of bounces
    """
    # Calculate distances between consecutive vertices
    segment_distances = []
    total_distance = 0.0
    depth = 0
    
    total_points = []
    total_points.append(tx_pos)
    
    i = 0
    while i < points.shape[0] and not all(np.isclose([0.0, 0.0, 0.0], points[i])):
        # drop invalid points(negative z, or bad objects)
        if points[i, 2] >= 0 and objects[i] != 4294967295 :
            total_points.append(points[i])
        
        i += 1
    total_points.append(rx_pos)
    # print(total_points)
    total_points = np.asarray(total_points)
        
    for j in range(len(total_points) - 1):
        segment_dist = np.linalg.norm(total_points[j+1, :] - total_points[j, :])
        segment_distances.append(segment_dist)
        total_distance += segment_dist
        depth += 1

    # print("Total path distance:", total_distance)    
    depth = depth - 1
    return total_distance, depth

In [43]:
# Borrowed from Sionna report

# Calculate angle between two vectors
def angle_between_vectors(v1, v2):
    """Calculate the angle in degrees between two vectors."""
    v1_u = v1 / np.linalg.norm(v1)
    v2_u = v2 / np.linalg.norm(v2)
    dot_product = np.clip(np.dot(v1_u, v2_u), -1.0, 1.0)
    # print("Dot product:", dot_product)
    angle_rad = np.arccos(dot_product)
    # print("Angle in radians:", angle_rad)
    angle_deg = np.degrees(angle_rad)
    # print("Angle in degrees:", angle_deg)
    return angle_deg

# Calculate Fresnel reflection coefficient
def fresnel_reflection_coefficient(epsilon_r, sigma, fc, epsilon_0, incident_angle_deg):
    """Calculate the Fresnel reflection coefficient for perpendicular polarization."""
    
    epsilon_c = epsilon_r - 1j * sigma / (2 * np.pi * fc * epsilon_0)
#     print("Hand Complex permittivity:", epsilon_c)
    incident_angle_rad = np.radians(incident_angle_deg)

    Z = np.sqrt(epsilon_c-np.cos(incident_angle_rad)**2)
    R_parallel = (epsilon_c*np.sin(incident_angle_rad) - Z) / (epsilon_c*np.sin(incident_angle_rad) + Z)
    R_perpendicular = (np.sin(incident_angle_rad) - Z) / (np.sin(incident_angle_rad) + Z)
    # R = np.sqrt((np.abs(R_parallel)**2 + np.abs(R_perpendicular)**2) / 2)

    return R_perpendicular, R_parallel

# Convert reflection coefficient to normalized reflection loss in dB
def normalized_reflection_loss(G_t, G_r, R, d, fc):
    """Convert reflection coefficient to normalized reflection loss in dB."""

    c = 3e8  # Speed of light in m/s
    wavelength = c / fc
    RL = (G_t * G_r * (np.abs(R)**2)  * wavelength**2) / ((4 * np.pi * d)**2)
    RL_dB = -10 * np.log10(RL)
    
    return RL_dB

# Convert reflection coefficient to normalized reflection loss in dB
def normalized_scattered_reflection_loss(G_t, G_r, R, d, fc, ground_bounce_point, rx_pos, beam_angle):
    """Convert reflection coefficient to normalized reflection loss in dB."""

    s = 0.4
    source_rx = np.squeeze(rx_pos)
    outgoing_vec = source_rx - ground_bounce_point

    theta_o = angle_between_vectors(outgoing_vec, [0, 0, 1])  # degrees
    cos_term = np.abs(np.cos(np.radians(theta_o)))

    c = 3e8
    wavelength = c / fc

    delta_omega = 2 * np.pi * (1 - np.cos(beam_angle)) / 1_000_000
    

    # Diffuse probability
    p_diff = s

    RL = (
        G_t * G_r
        # * (1/p_diff)
        * delta_omega
        * (s**2)
        * (cos_term/ np.pi)
        * (2*np.abs(R))**2
        * wavelength**2
        / ((4 * np.pi * d)**2)
    )

    RL_dB = -10 * np.log10(RL)
    return RL_dB


In [13]:
friis = lambda d, f: 20*np.log10(d) + 20*np.log10(f) - 147.55

In [14]:
print(friis(250, 11.7e9))
print(friis(1000, 11.7e9))

for slant in slants:
    print(friis(slant, 11.7e9))

101.77251740836397
113.81371723492322
182.45321172157077
170.7022665997112
168.6209710248081


# 250 m Link Distance

In [7]:
rx_pos = [0,0,0.5]
tx_pos = [100, 0, 10]

angle = np.arctan(50/np.linalg.norm(tx_pos))

In [8]:
f = 11.7e9

sionna_taus_250 = []
sionna_amps_250 = []
sionna_verts_250 = []
sionna_obj_250 = []
sionna_int_250 = []

for i in range(1):
    
    lambda_c = 3e8 / f

    # small local area (~few wavelengths)
    d = lambda_c * 2

    rx_pos = [0 + np.random.uniform(-d, d),
                0 + np.random.uniform(-d, d),
                0.5]

    # Set concrete scattering properties
    scene = load_scene("./flat_plane/plain.xml", merge_shapes=False)
    scene.frequency = f  # 11.7 GHz (Ku-band downlink)
    mat = scene.get("itu_concrete")
    mat.scattering_coefficient = 0.4
    mat.scattering_pattern = LambertianPattern()

    # Create Transmitter
    scene.tx_array = PlanarArray(
        num_rows=1, # of rows
        num_cols=1, # of columns
        vertical_spacing=0.5, # in wavelengths
        horizontal_spacing=0.5, # in wavelengths
        pattern="iso", # Antenna pattern
        polarization="V" # Vertical polarization
    )

    tx = Transmitter(
        name="sat_tx",
        position=tx_pos
    )
    scene.add(tx)

    tx.power_dbm = 43 # Transmit power in dBm

    # Create Receiver
    scene.rx_array = PlanarArray(
        num_cols=1, 
        num_rows=1,
        vertical_spacing=0.5,
        horizontal_spacing=0.5,
        pattern="iso",
        polarization="V")


    rx = Receiver(
            name=f"ue_1",
            position= rx_pos,
            orientation=[0.0, 0.0, 0.0])
    scene.add(rx)  

    tx.look_at(rx_pos)  # Pointing to the center of the scene

#     # Maximize number of samples
#     num_samples = (2**32 - 1)//64 # 2^24 -1


    # Flush memory, 
    # was having problems with allocation on repeated runs
    for name in ["paths", "amp", "tau", "paths_buffer"]:
        if name in globals():
            del globals()[name]

    gc.collect()
    drjit.flush_malloc_cache()
    drjit.eval()

    p_solver  = PathSolver()

    # Compute propagation paths
    seed = drjit.cuda.ad.UInt(np.random.randint(1,100000))
    paths = p_solver(scene=scene,
                        max_depth=1,
                        los=True,
                        specular_reflection=False,
                        diffuse_reflection=True,
                        refraction=False,
                        diffraction=False,
                        synthetic_array=False,
                        max_num_paths_per_src=1_000_000,
                        samples_per_src=1_000_000,
                        seed=seed,
                        beamforming=True,
                        beam_angle=angle)
    amp, tau = paths.cir(normalize_delays=False, out_type="numpy")

    t = tau[0,0,0,0,:] #in seconds
    amp = np.abs(amp)[0,0,0,0,:,0]
    vertices = paths.vertices.numpy()
    objects = paths.objects.numpy()
    interactions = paths.interactions.numpy()

    # Shape: [max_depth, num_rx, num_rx_ant, num_tx, num_tx_ant, num_paths, 3]

    # Extract vertices for specific path
    vertices = vertices[:, 0, 0, 0, 0, :, :]
    objects = objects[:,0,0,0,:,:]

    sionna_taus_250.append(t)
    sionna_amps_250.append(amp)
    sionna_verts_250.append(vertices)
    sionna_obj_250.append(objects)
    sionna_int_250.append(interactions)



In [9]:
# Inspect number of paths found

print(len(sionna_taus_250[0]))

606173


In [10]:
# Calculate Mean and Variance
mean, variance = mean_and_variance(sionna_taus_250, sionna_amps_250)

print(mean)
print(variance)

[8.680806e-11]
[2.5254476e-09]


In [47]:
# Find LoS + Specular Paths
non_diff_idx = []

#loop through each elevation angle
for i in range(1):
    inter = sionna_int_250[i][:,0,0,0,0,:]
    # loop through each path
    temp_idx = []
    for j in range(inter.shape[1]):
        # check for diffuse or diffract reflection and append if none
        if (np.where(inter[:, j] == 2))[0].size == 0 and (np.where(inter[:, j] == 8))[0].size == 0:
            # append index of LoS or specular path
            temp_idx.append(j)
    non_diff_idx.append(temp_idx)

In [48]:
print(non_diff_idx)

[[16576]]


In [12]:
# Get dB amplitude

db_250 = [20*np.log10(i) for i in sionna_amps_250]

# Sorted weakest to strongest paths
#idx_20 = np.argsort(db_250[0])
#idx_50 = np.argsort(db_250[1])
#idx_90 = np.argsort(db_250[2])
# print(db_250[0][non_diff_idx[0]])

In [33]:
print(sionna_obj_250[0].shape)

(1, 1, 606173)


In [45]:
# Check if LoS and Ground Bounces are valid(ground bouncesnso we know normal vector is directly upward)
from scipy.constants import epsilon_0
non_diff_idx = [120, 10000]

for i in range(1):
    for index in non_diff_idx:
        dist, depth = calculate_path_distance(tx_pos, 
                                       rx_pos, 
                                       sionna_verts_250[i][:, index, :],
                                       sionna_obj_250[i][:, 0, index])
        
#         print(dist)
        #fspl = friis(dist, f)
#         print("FSPL:", fspl)
        print("Simulated Path Loss(dB):" , -db_250[i][index])
        #if not np.isclose(db_250[i][index], -fspl, atol=1):
        # calculate reflection loss(all ground bounces)
        ground_bounce_point = np.array(sionna_verts_250[i][0, index, :])
        source_tx = np.squeeze(tx_pos)
        incident_vec_p1 = source_tx - ground_bounce_point
        
        incident_angle = angle_between_vectors(incident_vec_p1, [0,0,1])
        
        
        epsilon_r_plane = mat.relative_permittivity[0]
        conductivity_plane = mat.conductivity[0]
        g = 1.0
        omega = 2 * np.pi * 11.7e9

        # fresnel_reflection_coefficient
        R_p1 = fresnel_reflection_coefficient(epsilon_r_plane, conductivity_plane, f, epsilon_0, 90-incident_angle)
        R = np.sqrt(0.5*(np.abs(R_p1[0])**2 + np.abs(R_p1[1])**2))
        
        RL_dB = normalized_scattered_reflection_loss(g, g, R, dist, f, ground_bounce_point, rx_pos, angle)
        # print("Path 1 distance:", path_distances[1]) #84.48353
        print(f"Reflected Path Loss(idx = {index}) (dB):", RL_dB)
        #else:
#             print()
            #print("LoS Path Loss (dB):", fspl)

Simulated Path Loss(dB): 200.98242
Reflected Path Loss(idx = 120) (dB): 189.41481281134554
Simulated Path Loss(dB): 197.74066
Reflected Path Loss(idx = 10000) (dB): 199.18590811924375


In [ ]:
fig, axes = plt.subplots(
    nrows=3,
    ncols=1,
    figsize=(8, 10)
)

cmap = mpl.colormaps['viridis']
C = cmap(np.linspace(0, 1, 6))

plt.subplot(3,1,1)


markerline, stemlines, baseline = plt.stem(sionna_taus_250[0][idx_20[-1000:]], 
                                           db_250[0][idx_20[-1000:]]-np.max(db_250[0])
                                           ,basefmt=" ")


plt.setp(markerline, color=C[0], markersize=6)
plt.setp(stemlines, color=C[0], linewidth=1.5)
plt.title(f"20 degree CIR(250 m)")
plt.grid()
plt.ylabel(r"Normalized Path Loss(dB)")
plt.xlabel(r"$\tau$ (s)")
plt.xlim(0.5e-6, 2.5e-6)

plt.subplot(3,1,2)
markerline, stemlines, baseline = plt.stem(sionna_taus_250[1][idx_50[-1000:]], 
                                           db_250[1][idx_50[-1000:]]-np.max(db_250[1]),
                                           basefmt=" ",)


plt.setp(markerline, color=C[3], markersize=6)
plt.setp(stemlines, color=C[3], linewidth=1.0)

plt.title(f"50 degree CIR(250 m)")
plt.grid()
plt.ylabel(r"Normalized Path Loss(dB)")
plt.xlabel(r"$\tau$ (s)")
plt.xlim(0.5e-6, 2.5e-6)

plt.subplot(3,1,3)

markerline, stemlines, baseline = plt.stem(sionna_taus_250[2][idx_90[-1000:]], 
                                           db_250[2][idx_90[-1000:]]-np.max(db_250[2])
                                           ,basefmt=" ")


plt.setp(markerline, color=C[5], markersize=6)
plt.setp(stemlines, color=C[5], linewidth=1.0)
# Color configuration
plt.title(f"90 degree CIR(250 m)")
plt.grid()
plt.ylabel(r"Normalized Path Loss(dB)")
plt.xlabel(r"$\tau$ (s)")
plt.xlim(0.5e-6, 2.5e-6)

# plt.xlim(800, 1200)

# plt.legend()
plt.tight_layout()
plt.show()

# 1km Link Distances

In [ ]:
sionna_taus_1k = []
sionna_amps_1k = []
sionna_verts_1k = []
sionna_obj_1k = []
sionna_int_1k = []

for i, tx_position in enumerate(sionna_1km):

    scene = load_scene("./wharf_scene_v3/canarywharf3.xml", merge_shapes=False)
    scene.frequency = f  # 11.7 GHz (Ku-band downlink)
    mat = scene.get("itu_concrete")
    mat.scattering_coefficient = 0.4
    mat.scattering_pattern = LambertianPattern()

    # Create Transmitter
    scene.tx_array = PlanarArray(
        num_rows=1, # of rows
        num_cols=1, # of columns
        vertical_spacing=0.5, # in wavelengths
        horizontal_spacing=0.5, # in wavelengths
        pattern="iso", # Antenna pattern
        polarization="V" # Vertical polarization
    )

    tx = Transmitter(
        name="sat_tx",
        position=tx_position
    )
    scene.add(tx)

    tx.power_dbm = 43 # Transmit power in dBm

    # Create Receiver
    scene.rx_array = PlanarArray(
        num_cols=1, 
        num_rows=1,
        vertical_spacing=0.5,
        horizontal_spacing=0.5,
        pattern="iso",
        polarization="V")


    rx = Receiver(
            name=f"ue_1",
            position= rx_pos_sionna,
            orientation=[0.0, 0.0, 0.0])
    scene.add(rx)  

    tx.look_at(rx_pos_sionna)  # Pointing to the center of the scene

#     # Maximize number of samples
#     num_samples = (2**32 - 1)//64 # 2^24 -1


    # Flush memory, having problems with allocation on repeated runs
    for name in ["paths", "amp", "tau", "paths_buffer"]:
        if name in globals():
            del globals()[name]

    gc.collect()
    drjit.flush_malloc_cache()
    drjit.eval()

    p_solver  = PathSolver()

    # Compute propagation paths
    seed = drjit.cuda.ad.UInt(np.random.randint(1,100000))
    paths = p_solver(scene=scene,
                        max_depth=5,
                        los=True,
                        specular_reflection=True,
                        diffuse_reflection=True,
                        refraction=False,
                        diffraction=True,
                        synthetic_array=False,
                        max_num_paths_per_src=1_000_000,
                        samples_per_src=10_000_000,
                        seed=seed,
                        beamforming=True,
                        beam_angle=angle_1k)
    amp, tau = paths.cir(normalize_delays=False, out_type="numpy")

    t = tau[0,0,0,0,:] #in seconds
    amp = np.abs(amp)[0,0,0,0,:,0]
    vertices = paths.vertices.numpy()
    objects = paths.objects.numpy()
    interactions = paths.interactions.numpy()

    # Shape: [max_depth, num_rx, num_rx_ant, num_tx, num_tx_ant, num_paths, 3]
    # vertices = paths.vertices.numpy()

    # Extract vertices for specific path
    vertices = vertices[:, 0, 0, 0, 0, :, :]
    objects = objects[:,0,0,0,:,:]

    sionna_taus_1k.append(t)
    sionna_amps_1k.append(amp)
    sionna_verts_1k.append(vertices)
    sionna_obj_1k.append(objects)
    sionna_int_1k.append(interactions)



In [ ]:
print(len(sionna_taus_1k[0]), len(sionna_taus_1k[1]), len(sionna_taus_1k[2]))
# print(sionna_verts_1k[0].shape)

In [ ]:
# Find mean and variance
mean, variance = mean_and_variance(sionna_taus_1k, sionna_amps_1k)

print(mean)
print(variance)

In [ ]:
# Find LoS + Specular Paths
non_diff_idx = []

#loop through each elevation angle
for i in range(3):
    inter = sionna_int_1k[i][:,0,0,0,0,:]
    # loop through each path
    temp_idx = []
    for j in range(inter.shape[1]):
        # check for diffuse or diffract reflection and append if none
        if (np.where(inter[:, j] == 2))[0].size == 0 and (np.where(inter[:, j] == 8))[0].size == 0:
            # append index of LoS or specular path
            temp_idx.append(j)
    non_diff_idx.append(temp_idx)



In [ ]:
db_1k = [20*np.log10(i) for i in sionna_amps_1k]

In [ ]:
# Check if LoS and Ground Bounces are valid(ground bounces good, so we know normal vector is directly upward)

for i in range(3):
    for index in non_diff_idx[i]:
        dist, depth = calculate_path_distance(sionna_1km[i], 
                                       rx_pos_sionna, 
                                       sionna_verts_1k[i][:, index, :],
                                       sionna_obj_1k[i][:, 0, index])
        
#         print(dist)
        fspl = friis(dist, f)
#         print("FSPL:", fspl)
        print("Simulated Path Loss(dB):" , -db_1k[i][index])
        if not np.isclose(db_1k[i][index], -fspl, atol=1):
            # calculate reflection loss(all ground bounces)
            ground_bounce_point = np.array(sionna_verts_1k[i][0, index, :])
            source_tx = np.squeeze(sionna_1km[i])
            incident_vec_p1 = source_tx - ground_bounce_point
            
            incident_angle = angle_between_vectors(incident_vec_p1, [0,0,1])
            
            
            epsilon_r_plane = mat.relative_permittivity[0]
            conductivity_plane = mat.conductivity[0]
            g = 1.0
            omega = 2 * np.pi * f

            # fresnel_reflection_coefficient

            R_p1 = fresnel_reflection_coefficient(epsilon_r_plane, conductivity_plane, f, epsilon_0, 90-incident_angle)
            
            RL_dB = normalized_reflection_loss(g, g, R_p1[1], dist, f)
            # print("Path 1 distance:", path_distances[1]) #84.48353
            print("Reflected Path Loss (dB):", RL_dB)
        else:
#             print()
            print("LoS Path Loss (dB):", fspl)

In [ ]:
idx_20 = np.argsort(db_1k[0])
idx_50 = np.argsort(db_1k[1])
idx_90 = np.argsort(db_1k[2])

fig, axes = plt.subplots(
    nrows=3,
    ncols=1,
    figsize=(8, 10)
)

cmap = mpl.colormaps['viridis']
C = cmap(np.linspace(0, 1, 6))

plt.subplot(3,1,1)


markerline, stemlines, baseline = plt.stem(sionna_taus_1k[0][idx_20], 
                                           db_1k[0][idx_20]-np.max(db_1k[0]),
                                           basefmt=" ", label='20 degree')


plt.setp(markerline, color=C[0], markersize=6)
plt.setp(stemlines, color=C[0], linewidth=1.5)
plt.title(f"20 degree CIR(1 km)")
plt.grid()
plt.ylabel(r"Path Loss(dB)")
plt.xlabel(r"$\tau$ (s)")
plt.xlim(3e-6, 10e-6)

plt.subplot(3,1,2)
markerline, stemlines, baseline = plt.stem(sionna_taus_1k[1][idx_50], 
                                           db_1k[1][idx_50]-np.max(db_1k[1]),
                                           basefmt=" ", label='50 degree')


plt.setp(markerline, color=C[3], markersize=6)
plt.setp(stemlines, color=C[3], linewidth=1.0)

plt.title(f"50 degree CIR(1 km)")
plt.grid()
plt.ylabel(r"Path Loss(dB)")
plt.xlabel(r"$\tau$ (s)")
plt.xlim(3e-6, 10e-6)

plt.subplot(3,1,3)

markerline, stemlines, baseline = plt.stem(sionna_taus_1k[2][idx_90], 
                                           db_1k[2][idx_90]-np.max(db_1k[2]),
                                           basefmt=" ", label='90 degree')


plt.setp(markerline, color=C[5], markersize=6)
plt.setp(stemlines, color=C[5], linewidth=1.0)
# Color configuration
plt.title(f"90 degree CIR(1 km)")
plt.grid()
plt.ylabel(r"Path Loss(dB)")
plt.xlabel(r"$\tau$ (s)")

plt.xlim(3e-6, 10e-6)

# plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# export path information
with open("sionna_cir_1k.pkl", "wb") as f:
    pickle.dump({"taus": sionna_taus_1k, "amps": sionna_amps_1k, 
                 "vertices": sionna_verts_1k, "objects": sionna_obj_1k,
                "interactions": sionna_int_1k}, f, protocol=pickle.HIGHEST_PROTOCOL)

# LEO(550 km) Link Distances

In [ ]:
sionna_taus_leo = []
sionna_amps_leo = []
sionna_verts_leo = []
sionna_obj_leo = []
sionna_int_leo = []

for i, tx_position in enumerate(sionna_LEO):

    scene = load_scene("./wharf_scene_v3/canarywharf3.xml", merge_shapes=False)
    scene.frequency = f  # 11.7 GHz (Ku-band downlink)
    mat = scene.get("itu_concrete")
    mat.scattering_coefficient = 0.4
    mat.scattering_pattern = LambertianPattern()

    # Create Transmitter
    scene.tx_array = PlanarArray(
        num_rows=1, # of rows
        num_cols=1, # of columns
        vertical_spacing=0.5, # in wavelengths
        horizontal_spacing=0.5, # in wavelengths
        pattern="iso", # Antenna pattern
        polarization="V" # Vertical polarization
    )

    tx = Transmitter(
        name="sat_tx",
        position=tx_position
    )
    scene.add(tx)

    tx.power_dbm = 43 # Transmit power in dBm

    # Create Receiver
    scene.rx_array = PlanarArray(
        num_cols=1, 
        num_rows=1,
        vertical_spacing=0.5,
        horizontal_spacing=0.5,
        pattern="iso",
        polarization="V")


    rx = Receiver(
            name=f"ue_1",
            position= rx_pos_sionna,
            orientation=[0.0, 0.0, 0.0])
    scene.add(rx)  

    tx.look_at(rx_pos_sionna)  # Pointing to the center of the scene

#     # Maximize number of samples
#     num_samples = (2**32 - 1)//256 # 2^24 -1


    # Flush memory, having problems with allocation on repeated runs
    for name in ["paths", "amp", "tau", "paths_buffer"]:
        if name in globals():
            del globals()[name]

    gc.collect()
    drjit.flush_malloc_cache()
    drjit.eval()

    p_solver  = PathSolver()

    # Compute propagation paths
    seed = drjit.cuda.ad.UInt(np.random.randint(1,100000))
    paths = p_solver(scene=scene,
                        max_depth=5,
                        los=True,
                        specular_reflection=True,
                        diffuse_reflection=True,
                        refraction=False,
                        diffraction=True,
                        synthetic_array=False,
                        max_num_paths_per_src=1_000_000,
                        samples_per_src=10_000_000,
                        seed=seed,
                        beamforming=True,
                        beam_angle=angle_leo)
    
    amp, tau = paths.cir(normalize_delays=False, out_type="numpy")

    t = tau[0,0,0,0,:] #in seconds
    amp = np.abs(amp)[0,0,0,0,:,0]
    vertices = paths.vertices.numpy()
    objects = paths.objects.numpy()
    interactions = paths.interactions.numpy()

    # Shape: [max_depth, num_rx, num_rx_ant, num_tx, num_tx_ant, num_paths, 3]
    # vertices = paths.vertices.numpy()

    # Extract vertices for specific path
    vertices = vertices[:, 0, 0, 0, 0, :, :]
    objects = objects[:,0,0,0,:,:]

    sionna_taus_leo.append(t)
    sionna_amps_leo.append(amp)
    sionna_verts_leo.append(vertices)
    sionna_obj_leo.append(objects)
    sionna_int_leo.append(interactions)


In [ ]:
# Inspect num paths
print(len(sionna_taus_leo[0]), len(sionna_taus_leo[1]), len(sionna_taus_leo[2]))

In [ ]:
mean, variance = mean_and_variance(sionna_taus_leo, sionna_amps_leo)

print(mean)
print(variance)

In [ ]:
# Find LoS + Specular Paths
non_diff_idx = []

#loop through each elevation angle
for i in range(3):
    inter = sionna_int_leo[i][:,0,0,0,0,:]
    # loop through each path
    temp_idx = []
    for j in range(inter.shape[1]):
        # check for diffuse or diffract reflection and append if none
        if (np.where(inter[:, j] == 2))[0].size == 0 and (np.where(inter[:, j] == 8))[0].size == 0:
            # append index of LoS or specular path
            temp_idx.append(j)
    non_diff_idx.append(temp_idx)



In [ ]:
db_leo = [20*np.log10(i) for i in sionna_amps_leo]

In [ ]:
# Making Sure LoS is still first path
time_sorted = np.argsort(sionna_taus_leo[0])
print(time_sorted[0])
print(non_diff_idx[0])



In [ ]:
# Check if LoS and Ground Bounces are valid(ground bounces good, so we know normal vector is directly upward)

for i in range(3):
    for index in non_diff_idx[i]:
        dist, depth = calculate_path_distance(sionna_LEO[i], 
                                       rx_pos_sionna, 
                                       sionna_verts_leo[i][:, index, :],
                                       sionna_obj_leo[i][:, 0, index])
        
#         print(dist)
        fspl = friis(dist, f)
#         print("FSPL:", fspl)
        print("Simulated Path Loss(dB):" , -db_leo[i][index])
        if not np.isclose(db_leo[i][index], -fspl, atol=1):
            # calculate reflection loss(all ground bounces)
            ground_bounce_point = np.array(sionna_verts_leo[i][0, index, :])
            source_tx = np.squeeze(sionna_LEO[i])
            incident_vec_p1 = source_tx - ground_bounce_point
            
            incident_angle = angle_between_vectors(incident_vec_p1, [0,0,1])
            
            
            epsilon_r_plane = mat.relative_permittivity[0]
            conductivity_plane = mat.conductivity[0]
            g = 1.0
            omega = 2 * np.pi * f

            # fresnel_reflection_coefficient

            R_p1 = fresnel_reflection_coefficient(epsilon_r_plane, conductivity_plane, f, epsilon_0, 90-incident_angle)
            # print("Hand Path 1 Fresnel Reflection Coefficient:", R_p1)

            # Path 1 Path Loss
#             print(dist)
            RL_dB = normalized_reflection_loss(g, g, R_p1[1], dist, f)
            # print("Path 1 distance:", path_distances[1]) #84.48353
            print("Reflected Path Loss (dB):", RL_dB)
        else:
#             print()
            print("LoS Path Loss (dB):", fspl)

In [ ]:
idx_20 = np.argsort(db_leo[0])
idx_50 = np.argsort(db_leo[1])
idx_90 = np.argsort(db_leo[2])

fig, axes = plt.subplots(
    nrows=3,
    ncols=1,
    figsize=(8, 10)
)

cmap = mpl.colormaps['viridis']
C = cmap(np.linspace(0, 1, 6))

plt.subplot(3,1,1)


markerline, stemlines, baseline = plt.stem(sionna_taus_leo[0][idx_20[-10000:]], 
                                           db_leo[0][idx_20[-10000:]]-np.max(db_leo[0]),
                                           basefmt=" ", label='Strongest Paths')


plt.setp(markerline, color=C[0], markersize=6)
plt.setp(stemlines, color=C[0], linewidth=1.5)

los_idx = np.argmin(sionna_taus_leo[0])
markerline, stemlines, baseline = plt.stem(sionna_taus_leo[0][los_idx], 
                                           db_leo[0][los_idx]-np.max(db_leo[0]),
                                           basefmt=" ", label='LoS')


plt.setp(markerline, color=C[5], markersize=6)
plt.setp(stemlines, color=C[5], linewidth=1.0)

plt.title(f"20 degree CIR(LEO)")
plt.grid()
plt.ylabel(r"Path Loss(dB)")
plt.xlabel(r"$\tau$ (s)")
plt.legend()
# plt.xlim(9.017e-3, 9.019e-3)

plt.subplot(3,1,2)
markerline, stemlines, baseline = plt.stem(sionna_taus_leo[1][idx_50[-10000:]], 
                                           db_leo[1][idx_50[-10000:]]-np.max(db_leo[1]),
                                           basefmt=" ", label='Strongest Paths')


plt.setp(markerline, color=C[0], markersize=6)
plt.setp(stemlines, color=C[0], linewidth=1.0)

los_idx = np.argmin(sionna_taus_leo[1])
markerline, stemlines, baseline = plt.stem(sionna_taus_leo[1][los_idx], 
                                           db_leo[1][los_idx]-np.max(db_leo[1]),
                                           basefmt=" ", label='LoS')


plt.setp(markerline, color=C[5], markersize=6)
plt.setp(stemlines, color=C[5], linewidth=1.0)

plt.title(f"50 degree CIR(LEO)")
plt.grid()
plt.ylabel(r"Path Loss(dB)")
plt.xlabel(r"$\tau$ (s)")
plt.legend()
# plt.xlim(1.5e-3, 10e-3)

plt.subplot(3,1,3)

markerline, stemlines, baseline = plt.stem(sionna_taus_leo[2][idx_90[-10000:]], 
                                           db_leo[2][idx_90[-10000:]]-np.max(db_leo[2]),
                                           basefmt=" ", label='Strongest Paths')


plt.setp(markerline, color=C[0], markersize=6)
plt.setp(stemlines, color=C[0], linewidth=1.0)

los_idx = np.argmin(sionna_taus_leo[2])
markerline, stemlines, baseline = plt.stem(sionna_taus_leo[2][los_idx], 
                                           db_leo[2][los_idx]-np.max(db_leo[2]),
                                           basefmt=" ", label='LoS')


plt.setp(markerline, color=C[5], markersize=6)
plt.setp(stemlines, color=C[5], linewidth=1.0)
# Color configuration
plt.title(f"90 degree CIR(LEO)")
plt.grid()
plt.ylabel(r"Path Loss(dB)")
plt.xlabel(r"$\tau$ (s)")

# plt.xlim(1.5e-3, 10e-3)

plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# export path information
with open("sionna_cir_leo.pkl", "wb") as f:
    pickle.dump({"taus": sionna_taus_leo, "amps": sionna_amps_leo, 
                 "vertices": sionna_verts_leo, "objects": sionna_obj_leo,
                "interactions": sionna_int_leo}, f, protocol=pickle.HIGHEST_PROTOCOL)